In [ ]:
import pandas as pd

df = pd.read_csv("articles.csv")
print(df["product_type_name"].nunique())
print(df["product_type_name"].unique())  #Show unique product types to decide price ranges


131
['Vest top' 'Bra' 'Underwear Tights' 'Socks' 'Leggings/Tights' 'Sweater'
 'Top' 'Trousers' 'Hair clip' 'Umbrella' 'Pyjama jumpsuit/playsuit'
 'Bodysuit' 'Hair string' 'Unknown' 'Hoodie' 'Sleep Bag' 'Hair/alice band'
 'Belt' 'Boots' 'Bikini top' 'Swimwear bottom' 'Underwear bottom'
 'Swimsuit' 'Skirt' 'T-shirt' 'Dress' 'Hat/beanie' 'Kids Underwear top'
 'Shorts' 'Shirt' 'Cap/peaked' 'Pyjama set' 'Sneakers' 'Sunglasses'
 'Cardigan' 'Gloves' 'Earring' 'Bag' 'Blazer' 'Other shoe'
 'Jumpsuit/Playsuit' 'Sandals' 'Jacket' 'Costumes' 'Robe' 'Scarf' 'Coat'
 'Other accessories' 'Polo shirt' 'Slippers' 'Night gown' 'Alice band'
 'Straw hat' 'Hat/brim' 'Tailored Waistcoat' 'Necklace' 'Ballerinas' 'Tie'
 'Pyjama bottom' 'Felt hat' 'Bracelet' 'Blouse' 'Outdoor overall' 'Watch'
 'Underwear body' 'Beanie' 'Giftbox' 'Sleeping sack' 'Dungarees'
 'Outdoor trousers' 'Wallet' 'Swimwear set' 'Swimwear top' 'Flat shoe'
 'Garment Set' 'Ring' 'Waterbottle' 'Wedge' 'Long John'
 'Outdoor Waistcoat' 'Pumps' '

In [3]:
# Impute price & stock by product_type_name 
import numpy as np
import pandas as pd

# Reproducible randomness
rng = np.random.default_rng(42)
def r(low, high):  # random float
    return float(rng.uniform(low, high))
def s(low, high):  # random int
    return int(rng.integers(low, high))

def impute_price_stock(row):
    t = str(row["product_type_name"]).lower()

    # Tops / upper body
    if any(x in t for x in ["t-shirt","top","blouse","shirt","polo","hoodie","cardigan","sweater","vest","bodysuit"]):
        return pd.Series([round(r(200, 900), 2), s(50, 200)])

    # Bottoms
    elif any(x in t for x in ["trousers","jeans","shorts","skirt","leggings","tights","long john","dungarees","outdoor trousers"]):
        return pd.Series([round(r(300, 900), 2), s(40, 150)])

    # Dresses / sets
    elif any(x in t for x in ["dress","playsuit","jumpsuit","garment set","robe","night gown"]):
        return pd.Series([round(r(500, 1400), 2), s(40, 120)])

    # Underwear / sleep / swim
    elif any(x in t for x in ["underwear","bra","pyjama","bikini","swim","sleep","night","corset","underdress","sleeping sack","sleep bag"]):
        return pd.Series([round(r(200, 500), 2), s(100, 400)])

    # Footwear
    elif any(x in t for x in ["boot","sneaker","sandal","pump","heel","ballerina","flat shoe","flat shoes",
                              "flip flop","moccasin","pre-walker","other shoe","slipper","wedge","pumps","bootie"]):
        return pd.Series([round(r(700, 1800), 2), s(30, 120)])  

    # Accessories (hats, scarves, gloves, hair, ties, braces)
    elif any(x in t for x in ["belt","hat","cap","scarf","glove","tie","brace","hair","headband","alice",
                              "beanie","bucket hat","straw hat","felt hat","bucket"]):
        return pd.Series([round(r(200, 600), 2), s(50, 300)])

    # Jewelry & watches
    elif any(x in t for x in ["necklace","bracelet","ring","earring","earrings","watch"]):
        return pd.Series([round(r(300, 1800), 2), s(30, 200)])

    # Bags & wallets
    elif any(x in t for x in ["bag","wallet","cross-body","bumbag","tote","weekend","shoulder","backpack"]):
        return pd.Series([round(r(400, 1800), 2), s(30, 150)])

    # Cosmetics & care
    elif any(x in t for x in ["cosmetic","mist","spray","stain"]):
        return pd.Series([round(r(200, 600), 2), s(40, 200)])

    # Toys & home
    elif any(x in t for x in ["toy","blanket","cushion","towel","table","marker","wood","keychain",
                              "waterbottle","side table"]):
        return pd.Series([round(r(200, 800), 2), s(30, 100)])

    # Pets
    elif "dog" in t:
        return pd.Series([round(r(200, 700), 2), s(30, 50)])  

    # Tech accessories
    elif any(x in t for x in ["mobile","earphone","eyeglass","eyeglasses","wireless"]):
        return pd.Series([round(r(200, 700), 2), s(30, 100)]) 

    # Fallback
    else:
        return pd.Series([round(r(200, 600), 2), s(30, 150)]) 

# Apply to dataset
df = pd.read_csv("articles.csv")
# Impute missing detail_desc with default text
df["detail_desc"] = df["detail_desc"].fillna("No description available")

# Impute price and stock
df[["price","stock"]] = df.apply(impute_price_stock, axis=1)

# Ensure absolutely no stock below 30 
df["stock"] = df["stock"].clip(lower=30)

# Align with Oracle db schema
df = df.rename(columns={"article_id": "external_article_id"})

# Save result
df.to_csv("articles_price.csv", index=False, encoding="utf-8")
print("Saved: articles_imputed_realistic.csv")


Saved: articles_imputed_realistic.csv


In [4]:
# Validation
import pandas as pd

# Load file
df = pd.read_csv("articles_price.csv")

# Check for missing or invalid values
missing_price = df["price"].isna().sum()
missing_stock = df["stock"].isna().sum()

# Check for values below thresholds
below_price = (df["price"] < 200).sum()
below_stock = (df["stock"] < 30).sum()

# Print summary
print("=== Validation Summary ===")
print(f"Total rows: {len(df):,}")
print(f"Missing prices: {missing_price}")
print(f"Missing stocks: {missing_stock}")
print(f"Prices below 200: {below_price}")
print(f"Stocks below 30: {below_stock}")

# Confirm all are valid
if missing_price == 0 and missing_stock == 0 and below_price == 0 and below_stock == 0:
    print("All products have valid price and stock values.")
else:
    print("Some issues detected. Please review the counts above.")


=== Validation Summary ===
Total rows: 105,542
Missing prices: 0
Missing stocks: 0
Prices below 200: 0
Stocks below 30: 0
All products have valid price and stock values.
